# 📦 Monthly Data Extraction & Relationship Validation
This notebook extracts **one month of data** from 14 Parquet tables and validates all inter-table relationships to ensure a **100% clean development dataset**.

---
### Workflow
1. **Configuration** – Define paths, table names, date column, and relationships
2. **Load Full Tables** – Read all 14 parquet files
3. **Filter by Month** – Extract rows matching the target month
4. **Relationship Validation** – Validate every FK → PK link
5. **Fix Orphans** – Remove rows that break referential integrity (cascading)
6. **Final QA Report** – Row counts, null checks, uniqueness checks
7. **Export** – Save clean monthly parquet files

In [1]:
# ─────────────────────────────────────────────
# Cell 1 | Install / Import Dependencies
# ─────────────────────────────────────────────
# Uncomment the line below if running for the first time
# !pip install pandas pyarrow fastparquet tqdm tabulate openpyxl --quiet

import pandas as pd
import pyarrow.parquet as pq
import os
import json
import warnings
from pathlib import Path
from datetime import datetime
from tqdm.notebook import tqdm
from tabulate import tabulate
from IPython.display import display, HTML

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:,.2f}'.format)

print("✅ All libraries loaded successfully.")

✅ All libraries loaded successfully.


In [ ]:
# ─────────────────────────────────────────────
# Cell 2 | ⚙️ CONFIGURATION — Edit This Section
# ─────────────────────────────────────────────

# ── 1. Paths ──────────────────────────────────
INPUT_DIR  = Path("./data/parquet")      # Folder containing all 14 .parquet files
OUTPUT_DIR = Path("./data/monthly_dev")  # Folder where clean monthly files are saved
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── 2. Target Month ───────────────────────────
TARGET_YEAR  = 2024
TARGET_MONTH = 3          # e.g. 3 = March

# ── 3. Table Registry ─────────────────────────
# Map: logical_name → { file, date_col, pk }
# Adjust file names, date columns, and primary keys to match YOUR schema
TABLE_CONFIG = {
    "customers":       {"file": "customers.parquet",        "date_col": "created_at",     "pk": "customer_id"},
    "orders":          {"file": "orders.parquet",           "date_col": "order_date",      "pk": "order_id"},
    "order_items":     {"file": "order_items.parquet",      "date_col": "created_at",      "pk": "order_item_id"},
    "products":        {"file": "products.parquet",         "date_col": None,             "pk": "product_id"},
    "categories":      {"file": "categories.parquet",       "date_col": None,             "pk": "category_id"},
    "suppliers":       {"file": "suppliers.parquet",        "date_col": None,             "pk": "supplier_id"},
    "inventory":       {"file": "inventory.parquet",        "date_col": "updated_at",      "pk": "inventory_id"},
    "payments":        {"file": "payments.parquet",         "date_col": "payment_date",    "pk": "payment_id"},
    "shipments":       {"file": "shipments.parquet",        "date_col": "shipped_date",    "pk": "shipment_id"},
    "returns":         {"file": "returns.parquet",          "date_col": "return_date",     "pk": "return_id"},
    "reviews":         {"file": "reviews.parquet",          "date_col": "review_date",     "pk": "review_id"},
    "employees":       {"file": "employees.parquet",        "date_col": None,             "pk": "employee_id"},
    "stores":          {"file": "stores.parquet",           "date_col": None,             "pk": "store_id"},
    "promotions":      {"file": "promotions.parquet",       "date_col": "start_date",      "pk": "promotion_id"},
}

# ── 4. Relationships (FK → PK) ─────────────────
# Format: (child_table, fk_column, parent_table, pk_column)
RELATIONSHIPS = [
    ("orders",      "customer_id",   "customers",  "customer_id"),
    ("orders",      "store_id",      "stores",     "store_id"),
    ("orders",      "employee_id",   "employees",  "employee_id"),
    ("orders",      "promotion_id",  "promotions", "promotion_id"),
    ("order_items", "order_id",      "orders",     "order_id"),
    ("order_items", "product_id",    "products",   "product_id"),
    ("products",    "category_id",   "categories", "category_id"),
    ("products",    "supplier_id",   "suppliers",  "supplier_id"),
    ("inventory",   "product_id",    "products",   "product_id"),
    ("payments",    "order_id",      "orders",     "order_id"),
    ("shipments",   "order_id",      "orders",     "order_id"),
    ("returns",     "order_item_id", "order_items","order_item_id"),
    ("reviews",     "product_id",    "products",   "product_id"),
    ("reviews",     "customer_id",   "customers",  "customer_id"),
]

# ── 5. Nullable FK columns (allowed to be NULL without being an orphan) ──
NULLABLE_FKS = {
    ("orders", "promotion_id"),   # Not every order has a promotion
    ("orders", "employee_id"),    # Walk-in orders may have no employee
}

print(f"✅ Configuration loaded — Target: {TARGET_YEAR}-{TARGET_MONTH:02d}")
print(f"   Tables     : {len(TABLE_CONFIG)}")
print(f"   Relationships: {len(RELATIONSHIPS)}")

In [ ]:
# ─────────────────────────────────────────────
# Cell 3 | Load All 14 Parquet Tables
# ─────────────────────────────────────────────

raw_tables = {}  # logical_name → DataFrame (full data)
load_summary = []

for name, cfg in tqdm(TABLE_CONFIG.items(), desc="Loading tables"):
    filepath = INPUT_DIR / cfg["file"]
    if not filepath.exists():
        print(f"⚠️  File not found: {filepath}  →  skipping '{name}'")
        load_summary.append([name, cfg["file"], "❌ NOT FOUND", "-", "-"])
        continue

    df = pd.read_parquet(filepath)

    # Parse date column if present
    if cfg["date_col"] and cfg["date_col"] in df.columns:
        df[cfg["date_col"]] = pd.to_datetime(df[cfg["date_col"]], errors="coerce")

    raw_tables[name] = df
    load_summary.append([
        name, cfg["file"], "✅ OK",
        f"{len(df):,}", f"{df.shape[1]}"
    ])

print("\n" + tabulate(
    load_summary,
    headers=["Table", "File", "Status", "Rows", "Cols"],
    tablefmt="rounded_outline"
))

In [ ]:
# ─────────────────────────────────────────────
# Cell 4 | Filter Tables to Target Month
# ─────────────────────────────────────────────

monthly_tables = {}   # logical_name → monthly DataFrame
filter_summary = []

for name, df in raw_tables.items():
    cfg = TABLE_CONFIG[name]
    date_col = cfg["date_col"]

    if date_col and date_col in df.columns:
        # Filter to target month
        mask = (
            (df[date_col].dt.year  == TARGET_YEAR) &
            (df[date_col].dt.month == TARGET_MONTH)
        )
        filtered = df[mask].copy()
        filter_type = f"date filter on '{date_col}'"
    else:
        # Dimension / reference table — keep ALL rows (will be trimmed in validation)
        filtered = df.copy()
        filter_type = "full table (reference/dimension)"

    monthly_tables[name] = filtered
    filter_summary.append([
        name,
        f"{len(df):,}",
        f"{len(filtered):,}",
        f"{len(filtered)/len(df)*100:.1f}%" if len(df) > 0 else "0%",
        filter_type
    ])

print(f"\n📅 Month filter: {TARGET_YEAR}-{TARGET_MONTH:02d}\n")
print(tabulate(
    filter_summary,
    headers=["Table", "Full Rows", "Monthly Rows", "Retention %", "Filter Applied"],
    tablefmt="rounded_outline"
))

In [ ]:
# ─────────────────────────────────────────────
# Cell 5 | Relationship Validation Engine
# ─────────────────────────────────────────────

def validate_relationships(tables: dict, relationships: list, nullable_fks: set) -> pd.DataFrame:
    """Check every FK → PK link and return a summary DataFrame."""
    results = []

    for (child, fk_col, parent, pk_col) in relationships:
        if child not in tables or parent not in tables:
            results.append({
                "relationship": f"{child}.{fk_col} → {parent}.{pk_col}",
                "child_rows": "N/A", "orphan_count": "N/A",
                "orphan_%": "N/A", "status": "⚠️ TABLE MISSING"
            })
            continue

        child_df  = tables[child]
        parent_df = tables[parent]

        if fk_col not in child_df.columns:
            results.append({
                "relationship": f"{child}.{fk_col} → {parent}.{pk_col}",
                "child_rows": len(child_df), "orphan_count": "N/A",
                "orphan_%": "N/A", "status": "⚠️ FK COL MISSING"
            })
            continue

        parent_keys = set(parent_df[pk_col].dropna().unique())
        is_nullable = (child, fk_col) in nullable_fks

        if is_nullable:
            # Ignore NULL FK values — they are allowed
            child_fk = child_df[fk_col].dropna()
        else:
            child_fk = child_df[fk_col]

        orphan_mask  = ~child_fk.isin(parent_keys)
        orphan_count = int(orphan_mask.sum())
        total_rows   = len(child_fk)
        orphan_pct   = (orphan_count / total_rows * 100) if total_rows > 0 else 0

        status = "✅ VALID" if orphan_count == 0 else f"❌ {orphan_count} ORPHANS"

        results.append({
            "relationship":  f"{child}.{fk_col} → {parent}.{pk_col}",
            "child_rows":    total_rows,
            "orphan_count":  orphan_count,
            "orphan_%":      f"{orphan_pct:.2f}%",
            "nullable_fk":   "Yes" if is_nullable else "No",
            "status":        status
        })

    return pd.DataFrame(results)


print("🔍 Running initial relationship validation on monthly extract...\n")
validation_before = validate_relationships(monthly_tables, RELATIONSHIPS, NULLABLE_FKS)
display(validation_before.style.applymap(
    lambda v: "background-color: #d4edda" if "VALID" in str(v)
    else ("background-color: #f8d7da" if "ORPHAN" in str(v) else ""),
    subset=["status"]
))

total_issues = validation_before["orphan_count"].apply(
    lambda x: int(x) if str(x).isdigit() else 0
).sum()
print(f"\n⚠️  Total orphan records found: {total_issues:,}")

In [ ]:
# ─────────────────────────────────────────────
# Cell 6 | Auto-Fix: Remove Orphan Records (Cascading)
# ─────────────────────────────────────────────

def fix_orphans(tables: dict, relationships: list, nullable_fks: set, max_passes: int = 5) -> dict:
    """
    Iteratively remove orphan records until no FK violations remain.
    Cascades automatically: removing a parent row may orphan its children.
    """
    tables = {k: v.copy() for k, v in tables.items()}  # Don't mutate originals
    fix_log = []

    for pass_num in range(1, max_passes + 1):
        any_removed = False

        for (child, fk_col, parent, pk_col) in relationships:
            if child not in tables or parent not in tables:
                continue
            if fk_col not in tables[child].columns:
                continue

            parent_keys   = set(tables[parent][pk_col].dropna().unique())
            is_nullable   = (child, fk_col) in nullable_fks
            child_df      = tables[child]

            if is_nullable:
                orphan_mask = child_df[fk_col].notna() & ~child_df[fk_col].isin(parent_keys)
            else:
                orphan_mask = ~child_df[fk_col].isin(parent_keys)

            n_removed = int(orphan_mask.sum())
            if n_removed > 0:
                tables[child] = child_df[~orphan_mask].copy()
                fix_log.append({
                    "pass": pass_num,
                    "child_table": child,
                    "fk_col": fk_col,
                    "parent_table": parent,
                    "rows_removed": n_removed
                })
                any_removed = True

        if not any_removed:
            print(f"✅ Converged after {pass_num} pass(es) — no more orphans.")
            break
    else:
        print(f"⚠️  Max passes ({max_passes}) reached. Manual review may be needed.")

    return tables, pd.DataFrame(fix_log)


print("🔧 Fixing orphan records (cascading)...\n")
clean_tables, fix_log_df = fix_orphans(monthly_tables, RELATIONSHIPS, NULLABLE_FKS)

if not fix_log_df.empty:
    print("\n🗑️  Removal Log:")
    display(fix_log_df)
    print(f"\n   Total rows removed across all passes: {fix_log_df['rows_removed'].sum():,}")
else:
    print("   No rows needed removal — dataset was already clean! 🎉")

In [ ]:
# ─────────────────────────────────────────────
# Cell 7 | Post-Fix Validation (Should be 100% Clean)
# ─────────────────────────────────────────────

print("🔍 Running post-fix relationship validation...\n")
validation_after = validate_relationships(clean_tables, RELATIONSHIPS, NULLABLE_FKS)
display(validation_after.style.applymap(
    lambda v: "background-color: #d4edda" if "VALID" in str(v)
    else ("background-color: #f8d7da" if "ORPHAN" in str(v) else ""),
    subset=["status"]
))

remaining_issues = validation_after["orphan_count"].apply(
    lambda x: int(x) if str(x).isdigit() else 0
).sum()

if remaining_issues == 0:
    print("\n🏆 RESULT: 100% Referential Integrity Achieved!")
else:
    print(f"\n⚠️  WARNING: {remaining_issues} orphan(s) still remain — investigate manually.")

In [ ]:
# ─────────────────────────────────────────────
# Cell 8 | Data Quality Checks Per Table
# ─────────────────────────────────────────────

def qa_check_table(name: str, df: pd.DataFrame, cfg: dict) -> dict:
    pk = cfg["pk"]
    issues = []

    # 1. PK uniqueness
    if pk in df.columns:
        dupes = int(df[pk].duplicated().sum())
        if dupes > 0:
            issues.append(f"PK has {dupes} duplicate(s)")

    # 2. PK nulls
    if pk in df.columns:
        pk_nulls = int(df[pk].isna().sum())
        if pk_nulls > 0:
            issues.append(f"PK has {pk_nulls} null(s)")

    # 3. Empty table
    if len(df) == 0:
        issues.append("Table is EMPTY")

    # 4. High null % columns (>50%)
    null_pct = df.isnull().mean()
    high_null = null_pct[null_pct > 0.5].index.tolist()
    if high_null:
        issues.append(f"High nulls (>50%) in: {high_null}")

    return {
        "table": name,
        "rows": f"{len(df):,}",
        "cols": df.shape[1],
        "pk": pk,
        "pk_unique": "✅" if pk not in df.columns or int(df[pk].duplicated().sum()) == 0 else "❌",
        "pk_no_nulls": "✅" if pk not in df.columns or int(df[pk].isna().sum()) == 0 else "❌",
        "issues": "; ".join(issues) if issues else "None"
    }


qa_results = []
for name, df in clean_tables.items():
    qa_results.append(qa_check_table(name, df, TABLE_CONFIG[name]))

qa_df = pd.DataFrame(qa_results)
print("📋 Data Quality Report — Clean Monthly Tables\n")
display(qa_df.style.applymap(
    lambda v: "background-color: #d4edda" if v == "None" else
              ("background-color: #f8d7da" if v not in ["✅", "None"] else ""),
    subset=["issues"]
))

In [ ]:
# ─────────────────────────────────────────────
# Cell 9 | Row Count Comparison: Before vs After
# ─────────────────────────────────────────────

comparison = []
for name in TABLE_CONFIG:
    before = len(monthly_tables.get(name, []))
    after  = len(clean_tables.get(name, []))
    removed = before - after
    pct_kept = (after / before * 100) if before > 0 else 0

    comparison.append({
        "table":        name,
        "before_fix":   f"{before:,}",
        "after_fix":    f"{after:,}",
        "rows_removed": f"{removed:,}",
        "% kept":       f"{pct_kept:.1f}%"
    })

cmp_df = pd.DataFrame(comparison)
print("📊 Row Count: Before vs After Orphan Removal\n")
print(tabulate(cmp_df, headers="keys", tablefmt="rounded_outline", showindex=False))

In [ ]:
# ─────────────────────────────────────────────
# Cell 10 | Null Analysis Per Table
# ─────────────────────────────────────────────

print("🔎 Null Value Analysis (top columns per table)\n")

for name, df in clean_tables.items():
    null_counts = df.isnull().sum()
    null_pct    = (null_counts / len(df) * 100).round(2)
    null_df = pd.DataFrame({"null_count": null_counts, "null_%": null_pct})
    null_df = null_df[null_df["null_count"] > 0].sort_values("null_%", ascending=False)

    if null_df.empty:
        print(f"  ✅ {name}: No nulls")
    else:
        print(f"  ⚠️  {name}:")
        display(null_df.head(10))

In [ ]:
# ─────────────────────────────────────────────
# Cell 11 | Export Clean Tables as Parquet
# ─────────────────────────────────────────────

export_summary = []
month_label = f"{TARGET_YEAR}_{TARGET_MONTH:02d}"

for name, df in tqdm(clean_tables.items(), desc="Exporting"):
    out_file = OUTPUT_DIR / f"{name}_{month_label}.parquet"
    df.to_parquet(out_file, index=False, engine="pyarrow", compression="snappy")
    size_kb = out_file.stat().st_size / 1024
    export_summary.append([name, str(out_file.name), f"{len(df):,}", f"{size_kb:.1f} KB"])

print(f"\n✅ Exported {len(export_summary)} tables to: {OUTPUT_DIR.resolve()}\n")
print(tabulate(
    export_summary,
    headers=["Table", "Output File", "Rows", "Size"],
    tablefmt="rounded_outline"
))

In [ ]:
# ─────────────────────────────────────────────
# Cell 12 | Save Validation Report as Excel
# ─────────────────────────────────────────────

report_path = OUTPUT_DIR / f"validation_report_{month_label}.xlsx"

with pd.ExcelWriter(report_path, engine="openpyxl") as writer:
    validation_before.to_excel(writer, sheet_name="Relationships_Before", index=False)
    validation_after.to_excel(writer,  sheet_name="Relationships_After",  index=False)
    cmp_df.to_excel(writer,            sheet_name="Row_Counts",            index=False)
    qa_df.to_excel(writer,             sheet_name="QA_Checks",             index=False)
    if not fix_log_df.empty:
        fix_log_df.to_excel(writer,    sheet_name="Fix_Log",               index=False)

print(f"📄 Validation report saved: {report_path.name}")

In [ ]:
# ─────────────────────────────────────────────
# Cell 13 | 🏁 Final Summary
# ─────────────────────────────────────────────

total_rows_clean = sum(len(df) for df in clean_tables.values())
total_rows_raw   = sum(len(df) for df in monthly_tables.values())
valid_rels       = (validation_after["status"].str.contains("VALID")).sum()

summary_html = f"""
<div style="font-family:sans-serif; border:1px solid #ccc; border-radius:8px; padding:20px; max-width:600px;">
  <h2 style="color:#2c7be5;">🏁 Monthly Extract Summary</h2>
  <table style="width:100%; border-collapse:collapse;">
    <tr><td><b>Target Period</b></td><td>{TARGET_YEAR}-{TARGET_MONTH:02d}</td></tr>
    <tr><td><b>Tables Processed</b></td><td>{len(clean_tables)} / {len(TABLE_CONFIG)}</td></tr>
    <tr><td><b>Relationships Validated</b></td><td>{len(RELATIONSHIPS)}</td></tr>
    <tr><td><b>Valid Relationships</b></td><td style="color:green;"><b>{valid_rels} / {len(RELATIONSHIPS)}</b></td></tr>
    <tr><td><b>Rows (pre-fix)</b></td><td>{total_rows_raw:,}</td></tr>
    <tr><td><b>Rows (post-fix / clean)</b></td><td style="color:green;"><b>{total_rows_clean:,}</b></td></tr>
    <tr><td><b>Orphans Removed</b></td><td style="color:{'red' if fix_log_df is not None and not fix_log_df.empty else 'green'};">{'0' if fix_log_df is None or fix_log_df.empty else str(fix_log_df['rows_removed'].sum())}</td></tr>
    <tr><td><b>Output Directory</b></td><td><code>{OUTPUT_DIR.resolve()}</code></td></tr>
    <tr><td><b>Integrity Status</b></td><td style="color:{'green' if remaining_issues == 0 else 'red'};"><b>{'✅ 100% VALID' if remaining_issues == 0 else f'❌ {remaining_issues} issues remain'}</b></td></tr>
  </table>
</div>
"""
display(HTML(summary_html))